# Lesson 03 — 计时，但不要把程序卡住 — 参考实现

先完成主 Notebook 的练习再阅读。此 Notebook 只展示可通过 exercise 契约的一种实现。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


运行代码格，使用同一个可交互面板验证。然后回到主 Notebook，对照自己的状态和边界处理。


In [ ]:
%%academy_lab 03-nonblocking-timer solution exercise pass
#pragma once
#include "academy/device.hpp"
#include <cstdint>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& board);
        void tick();

    private:
        Device& board_;
        bool initialized_ = false, last_ = false, running_ = false;
        std::uint32_t deadline_ = 0;
    };
}

namespace academy {
    Controller::Controller(Device& b) : board_(b) {
    }
    void Controller::tick() {
        bool p = board_.read_button() == Level::low;
        if (!initialized_) {
            initialized_ = true;
            last_ = p;
            board_.output.state = "idle";
            return;
        }
        const bool edge = p && !last_;
        last_ = p;
        if (edge) {
            if (running_) {
                running_ = false;
                board_.output.state = "cancelled";
                ++board_.output.events;
            } else {
                running_ = true;
                deadline_ = board_.input.now_ms + 3000;
                board_.output.state = "running";
                ++board_.output.events;
            }
        }
        if (running_ && board_.input.now_ms >= deadline_) {
            running_ = false;
            board_.output.state = "done";
        }
        if (running_) {
            board_.output.reading = deadline_ - board_.input.now_ms;
            board_.output.brightness = static_cast<int>(board_.output.reading / 30);
        } else {
            board_.output.reading = 0;
            board_.output.brightness = 0;
        }
        board_.output.led = running_;
    }
}
